In [ ]:
%pip install dragoneye-python numpy Pillow 'imageio[pyav]' imageio-ffmpeg

In [ ]:
import os
from dragoneye import NormalizedBbox
import numpy as np

from PIL import Image, ImageDraw, ImageFont
import imageio.v3 as iio
import zlib
from dragoneye import Dragoneye, Video
from dragoneye.models import (
    ClassificationVideoObjectPrediction,
    ClassificationPredictVideoResponse,
)
from pathlib import Path
from typing import cast

from numpy.typing import NDArray
from IPython.display import display  # pyright: ignore [reportUnknownVariableType]

In [ ]:
# Configuration - Update these paths for your use case
LOCAL_VIDEO_PATH = "/path/to/input.mp4"

# Get API key from environment variable for security
API_KEY = os.environ.get("DRAGONEYE_API_KEY")
if not API_KEY:
    raise ValueError(
        "Please set DRAGONEYE_API_KEY environment variable. "
        "For this notebook, you can use: os.environ['DRAGONEYE_API_KEY'] = 'your-key-here'"
    )

MODEL_NAME = "recognize_anything/model_name"


# Initialize client and load media
client = Dragoneye(api_key=API_KEY)
media = Video.from_path(LOCAL_VIDEO_PATH)

# Load or fetch predictions
prediction_result = await client.classification.predict_video(
    media=media,
    model_name=MODEL_NAME,
)

In [ ]:
# Display and styling constants
LABEL_FONT_SIZE = 16
LABEL_PADDING = 4
LABEL_ALPHA = 128  # 0-255 for PIL (0 = transparent, 255 = opaque)
BBOX_THICKNESS_RATIO = 0.003  # Ratio of frame dimension to bbox thickness
MIN_BBOX_THICKNESS = 2
SHADOW_OFFSET = 2
DEFAULT_FPS = 30.0


def generate_deterministic_color(name: str) -> tuple[int, int, int]:
    """
    Generate a deterministic RGB color from a category name using hashing.

    Uses CRC32 to ensure consistent colors across runs for the same category name.
    Colors are generated to avoid being too dark (minimum brightness per channel).

    Args:
        name: Category name to generate color for

    Returns:
        Tuple of (red, green, blue) values in range [80, 207] for PIL RGB format

    Example:
        >>> generate_deterministic_color("trash")
        (201, 165, 142)
    """
    COLOR_CHANNEL_MASK = 0x7F
    COLOR_CHANNEL_OFFSET = 80

    hash_value = zlib.crc32(name.encode())

    # Spread hash across RGB channels and ensure minimum brightness
    red = ((hash_value >> 14) & COLOR_CHANNEL_MASK) + COLOR_CHANNEL_OFFSET
    green = ((hash_value >> 7) & COLOR_CHANNEL_MASK) + COLOR_CHANNEL_OFFSET
    blue = ((hash_value >> 0) & COLOR_CHANNEL_MASK) + COLOR_CHANNEL_OFFSET

    return (red, green, blue)


def denormalize_bbox(
    normalized_bbox: NormalizedBbox,
    frame_width: int,
    frame_height: int,
) -> tuple[int, int, int, int]:
    """
    Convert normalized bbox coordinates [0, 1] to absolute pixel coordinates.

    Args:
        normalized_bbox: Bounding box with coordinates in [0, 1] range as [x1, y1, x2, y2]
        frame_width: Width of the frame in pixels
        frame_height: Height of the frame in pixels

    Returns:
        Tuple of (x1, y1, x2, y2) in absolute pixel coordinates, clamped to frame bounds

    Example:
        >>> denormalize_bbox([0.1, 0.2, 0.9, 0.8], 1920, 1080)
        (192, 216, 1728, 864)
    """
    x1 = int(round(normalized_bbox[0] * frame_width))
    y1 = int(round(normalized_bbox[1] * frame_height))
    x2 = int(round(normalized_bbox[2] * frame_width))
    y2 = int(round(normalized_bbox[3] * frame_height))

    # Clamp to frame boundaries
    x1 = max(0, min(frame_width - 1, x1))
    y1 = max(0, min(frame_height - 1, y1))
    x2 = max(0, min(frame_width - 1, x2))
    y2 = max(0, min(frame_height - 1, y2))

    return x1, y1, x2, y2


def format_prediction_label(prediction: ClassificationVideoObjectPrediction) -> str:
    """
    Format a prediction into a display label string.

    Args:
        prediction: Classification prediction object

    Returns:
        Formatted label string with category name and optional score

    Example:
        >>> format_prediction_label(prediction)
        "trash_bag 0.95"
    """
    category = prediction.category
    name = category.displayName or category.name or "object"
    score = category.score

    if isinstance(score, (int, float)):
        return f"{name} {score:.2f}"
    return name


def draw_label_on_image(
    draw: ImageDraw.ImageDraw,
    x: int,
    y: int,
    label: str,
    color: tuple[int, int, int],
    image_width: int,
    image_height: int,
    font: ImageFont.FreeTypeFont | ImageFont.ImageFont | None = None,
) -> None:
    """
    Draw a filled label box with text on an image using PIL.

    Label is positioned above the bounding box by default, or inside if too close to top edge.
    This function modifies the draw object in place.

    Args:
        draw: PIL ImageDraw object to draw on
        x: X coordinate of the label anchor point (top-left of bbox)
        y: Y coordinate of the label anchor point (top-left of bbox)
        label: Text to display
        color: RGB color tuple for the label background
        image_width: Width of the image
        image_height: Height of the image
        font: PIL Font object (optional, uses default if None)

    Example:
        >>> draw_label_on_image(draw, 100, 100, "trash", (255, 0, 0), 1920, 1080)
    """
    # Get text bounding box
    bbox = draw.textbbox((0, 0), label, font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]

    # Determine label position (above bbox or inside if not enough space)
    label_y_top = y - text_height - LABEL_PADDING * 2
    if label_y_top < 0:
        label_y_top = y + LABEL_PADDING

    label_x_end = min(image_width - 1, x + text_width + LABEL_PADDING * 2)
    label_y_end = min(image_height - 1, label_y_top + text_height + LABEL_PADDING * 2)

    # Draw semi-transparent filled rectangle for label background
    color_with_alpha = color + (LABEL_ALPHA,)
    draw.rectangle(
        [(x, label_y_top), (label_x_end, label_y_end)],
        fill=color_with_alpha,
    )

    # Draw text on top of background
    text_y = label_y_top + LABEL_PADDING
    draw.text(
        (x + LABEL_PADDING, text_y),
        label,
        fill=(0, 0, 0),  # Black text for contrast
        font=font,
    )


def draw_prediction_on_frame(
    frame: Image.Image,
    prediction: ClassificationVideoObjectPrediction,
) -> None:
    """
    Draw a single prediction (bbox + label) on a PIL Image frame.

    This function modifies the frame in place.

    Args:
        frame: PIL Image object (video frame)
        prediction: Classification prediction to visualize
    """
    frame_width, frame_height = frame.size

    # Create a drawing context with alpha support
    if frame.mode != "RGBA":
        frame_rgba = frame.convert("RGBA")
        overlay = Image.new("RGBA", frame.size, (255, 255, 255, 0))
        draw = ImageDraw.Draw(overlay)
    else:
        frame_rgba = frame
        overlay = Image.new("RGBA", frame.size, (255, 255, 255, 0))
        draw = ImageDraw.Draw(overlay)

    # Extract prediction information
    x1, y1, x2, y2 = denormalize_bbox(
        prediction.normalizedBbox, frame_width, frame_height
    )
    label = format_prediction_label(prediction)
    category_name = (
        prediction.category.displayName or prediction.category.name or "object"
    )
    color = generate_deterministic_color(category_name)

    # Calculate adaptive thickness based on frame size
    thickness = max(
        MIN_BBOX_THICKNESS,
        int(round(BBOX_THICKNESS_RATIO * max(frame_width, frame_height))),
    )

    font = ImageFont.load_default(size=LABEL_FONT_SIZE)

    # Draw bounding box with shadow for depth on the overlay
    # Shadow
    for i in range(thickness + SHADOW_OFFSET):
        draw.rectangle(
            [(x1 - i, y1 - i), (x2 + i, y2 + i)],
            outline=(0, 0, 0, 255),
            width=1,
        )

    # Main box
    for i in range(thickness):
        draw.rectangle(
            [(x1 - i, y1 - i), (x2 + i, y2 + i)],
            outline=color + (255,),
            width=1,
        )

    # Draw label
    draw_label_on_image(draw, x1, y1, label, color, frame_width, frame_height, font)

    # Composite the overlay onto the original frame
    if frame.mode != "RGBA":
        frame_rgba = frame.convert("RGBA")

    frame_rgba = Image.alpha_composite(frame_rgba, overlay)

    # Convert back to RGB and paste into original frame
    result = frame_rgba.convert("RGB")
    frame.paste(result)

In [ ]:
def map_predictions_to_frames(
    predictions_response: ClassificationPredictVideoResponse,
    fps: float,
    total_frames: int,
) -> dict[int, list[ClassificationVideoObjectPrediction]]:
    """
    Map timestamp-based predictions to frame indices.

    Args:
        predictions_response: Response containing predictions with timestamps
        fps: Frames per second of the video
        total_frames: Total number of frames in the video

    Returns:
        Dictionary mapping frame index to list of predictions for that frame

    Example:
        >>> predictions_by_frame = map_predictions_to_frames(response, 30.0, 900)
        >>> predictions_by_frame[0]  # Predictions for first frame
        [<ClassificationVideoObjectPrediction>, ...]
    """
    frame_duration_us = 1e6 / fps
    predictions_by_frame: dict[int, list[ClassificationVideoObjectPrediction]] = {}

    for (
        timestamp_us,
        predictions,
    ) in predictions_response.timestamp_us_to_predictions.items():
        frame_idx = int(round(timestamp_us / frame_duration_us))

        # Skip out-of-range frames
        if 0 <= frame_idx < total_frames:
            predictions_by_frame.setdefault(frame_idx, []).extend(predictions)

    return predictions_by_frame


def process_video_with_predictions(
    input_path: str,
    predictions_response: ClassificationPredictVideoResponse,
) -> None:
    """
    Process a video file and overlay prediction visualizations using imageio.

    Args:
        input_path: Path to input video file
        output_path: Path where output video will be saved
        predictions_response: Predictions to visualize on the video

    Raises:
        FileNotFoundError: If input video doesn't exist
        RuntimeError: If video cannot be opened or writer fails

    Example:
        >>> process_video_with_predictions(
        ...     "input.mp4",
        ...     "output.mp4",
        ...     prediction_result
        ... )
        Done. Wrote: /path/to/output.mp4
    """
    # Validate input
    if not Path(input_path).exists():
        raise FileNotFoundError(f"Could not find video at {input_path}")

    # Get video properties using imageio
    metadata = iio.immeta(input_path, plugin="pyav")
    fps = cast(float, metadata.get("fps", DEFAULT_FPS))

    # Read all frames to get total count and dimensions
    print("Reading video frames...")
    frames = cast(list[NDArray[np.uint8]], iio.imread(input_path, plugin="pyav"))
    frame_height, frame_width = frames[0].shape[:2]

    total_frames = len(frames)
    print(
        f"Video properties: {frame_width}x{frame_height} @ {fps} FPS, {total_frames} frames"
    )

    # Map predictions to frame indices
    predictions_by_frame = map_predictions_to_frames(
        predictions_response, fps, total_frames
    )
    print(f"Found predictions for {len(predictions_by_frame)} frames")

    # Process frames with predictions
    print("Processing frames with predictions...")
    processed_frames = []

    for frame_idx, frame_array in enumerate(frames):
        # Convert numpy array to PIL Image
        frame_pil = Image.fromarray(frame_array)

        # Apply predictions if they exist for this frame
        if frame_idx in predictions_by_frame:
            for prediction in predictions_by_frame[frame_idx]:
                draw_prediction_on_frame(frame_pil, prediction)
                display(frame_pil)

        # Convert back to numpy array for imageio
        processed_frames.append(np.array(frame_pil))

        # Progress indicator
        if (frame_idx + 1) % 100 == 0:
            print(f"Processed {frame_idx + 1}/{total_frames} frames...")

In [ ]:
# Process the video with predictions
process_video_with_predictions(
    LOCAL_VIDEO_PATH,
    prediction_result,
)